# Decision tree classifier

This notebook:

1. selects peak detectors from `algorithms.py` and/or trained `AlgorithmGraph` artifacts,
2. builds or loads the classifier input dataframe,
3. trains and evaluates a decision tree.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree


# The notebook works when Jupyter is started either in the repository root
# or directly inside the classifier directory.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "detector_engine").exists():
    if (REPO_ROOT.parent / "detector_engine").exists():
        REPO_ROOT = REPO_ROOT.parent
    else:
        raise RuntimeError(
            "Could not find detector_engine. Start Jupyter from the repository "
            "root or from the classifier directory."
        )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from classifier.input_to_classifier import build_classifier_input
from detector_engine import load_best_graph

try:
    import algorithms
except ModuleNotFoundError:
    # Compatibility with the previous capitalized filename.
    import Algorithms as algorithms

In [2]:
QRS_ANNOTATIONS = {
    "N", "L", "R", "V", "/", "A", "f", "F",
    "a", "J", "S", "e", "j", "E", "Q",
}

GROUPING_SCHEMES = {
    "current": {
        "N": {"N"},
        "A": {"A"},
        "V": {"V"},
        "R": {"R"},
        "L": {"L"},
        "/": {"/"},
        "Other": QRS_ANNOTATIONS - {"N", "A", "V", 'R', 'L', '/'},
    },

    "aami": {
        "N": {"N", "L", "R", "e", "j"},
        "S": {"A", "a", "J", "S"},
        "V": {"V", "E"},
        "F": {"F"},
        "Q": {"/", "f", "Q"},
    },
}


def group_annotations(annotation_series, scheme):
    if scheme not in GROUPING_SCHEMES:
        raise ValueError(
            f"Unknown grouping scheme: {scheme!r}. "
            f"Available schemes: {list(GROUPING_SCHEMES)}"
        )

    groups = GROUPING_SCHEMES[scheme]

    label_to_group = {
        annotation: group_name
        for group_name, annotations in groups.items()
        for annotation in annotations
    }

    # These are generated by ClassifierInputBuilder,
    # rather than originating from the ECG annotation file.
    label_to_group["FP"] = "FP"
    label_to_group["AMB"] = "AMB"

    annotations = annotation_series.astype(str)
    grouped = annotations.map(label_to_group)

    unknown = sorted(annotations[grouped.isna()].unique())

    if unknown:
        raise ValueError(
            f"Annotations not covered by scheme {scheme!r}: {unknown}"
        )

    return grouped

In [3]:
LABEL_GROUPING = "current"
# LABEL_GROUPING = "aami"

In [5]:

DATA_PATH_TRAIN = REPO_ROOT / "dataset" / "train.pkl"
DATA_PATH_VAL = REPO_ROOT / "dataset" / "val.pkl"
DATA_PATH_TEST = REPO_ROOT / "dataset" / "test.pkl"

CLASSIFIER_INPUT_PATH_TRAIN = (
    REPO_ROOT / "classifier" / "classifier_input_train.pickle"
)
CLASSIFIER_INPUT_PATH_VAL = (
    REPO_ROOT / "classifier" / "classifier_input_val.pickle"
)
CLASSIFIER_INPUT_PATH_TEST = (
    REPO_ROOT / "classifier" / "classifier_input_test.pickle"
)

REFERENCE_DETECTOR = "alg1"
MATCH_WINDOW = 50

BASE_DETECTORS = {
    "alg1": algorithms.alg1_spanish,
    "alg2": algorithms.alg4_polish_20210222,
    "alg3": algorithms.alg3_iranian,
    "alg4": algorithms.alg5_pan_tompkins,
    "alg5": algorithms.Alg5_Turkish,
}


## Configuration

In [6]:

from pathlib import Path

import pandas as pd
from IPython.display import display


DATA_PATHS = {
    "train": DATA_PATH_TRAIN,
    "val": DATA_PATH_VAL,
    "test": DATA_PATH_TEST,
}

CLASSIFIER_INPUT_PATHS = {
    "train": CLASSIFIER_INPUT_PATH_TRAIN,
    "val": CLASSIFIER_INPUT_PATH_VAL,
    "test": CLASSIFIER_INPUT_PATH_TEST,
}

source_dfs = {}
classifier_dfs = {}

for split_name in ["train", "val", "test"]:
    source_df = pd.read_pickle(DATA_PATHS[split_name])

    classifier_df = build_classifier_input(
        dataset=source_df,
        detectors=BASE_DETECTORS,
        reference_detector=REFERENCE_DETECTOR,
        window=MATCH_WINDOW,
        output_path=CLASSIFIER_INPUT_PATHS[split_name],
    )

    source_dfs[split_name] = source_df
    classifier_dfs[split_name] = classifier_df

    print(
        f"{split_name.capitalize()} classifier input: "
        f"{len(classifier_df):,} rows"
    )
    print(f"Saved to: {CLASSIFIER_INPUT_PATHS[split_name]}")
    display(classifier_df.head())
    display(classifier_df["annotation"].value_counts())

source_df_train = source_dfs["train"]
source_df_val = source_dfs["val"]
source_df_test = source_dfs["test"]

classifier_df_train = classifier_dfs["train"]
classifier_df_val = classifier_dfs["val"]
classifier_df_test = classifier_dfs["test"]

# Keep the original variable available for cells that inspect one dataframe.
classifier_df = classifier_df_train


Train classifier input: 55,686 rows
Saved to: /Users/jeremiasz/code/ekg-evo/new/classifier/classifier_input_train.pickle


,record,reference_peak,alg2,alg3,alg4,alg5,annotation
0,232,31,-100,-100,-100,-100,FP
1,232,188,-100,-100,35,-100,FP
2,232,430,-100,-100,18,-100,FP
3,232,506,-13,-8,-100,-100,R
4,232,751,-12,-100,25,32,A


annotation
N     35809
L      4607
FP     4254
R      4068
V      2822
/      2394
A      1511
J        81
f        54
F        43
a        28
Q         8
j         6
E         1
Name: count, dtype: int64

Val classifier input: 27,291 rows
Saved to: /Users/jeremiasz/code/ekg-evo/new/classifier/classifier_input_val.pickle


,record,reference_peak,alg2,alg3,alg4,alg5,annotation
0,228,88,-100,-100,25,5,FP
1,228,165,-13,-6,-100,-100,N
2,228,437,-13,-6,21,28,N
3,228,730,-13,-6,-12,27,N
4,228,1065,-13,-6,3,27,N


annotation
N     18513
L      2003
V      1937
/      1283
R      1249
FP     1113
f       654
F       365
A       128
a        23
Q        23
Name: count, dtype: int64

Test classifier input: 28,806 rows
Saved to: /Users/jeremiasz/code/ekg-evo/new/classifier/classifier_input_test.pickle


,record,reference_peak,alg2,alg3,alg4,alg5,annotation
0,201,92,-100,-100,29,-100,FP
1,201,168,-100,-7,-47,-100,N
2,201,423,-100,-8,22,-100,N
3,201,694,-100,-8,1,28,N
4,201,913,-13,-7,-12,28,N


annotation
N     18955
V      2021
R      1910
/      1542
L      1456
FP      974
A       880
F       386
f       259
j       221
E       104
a        76
e        16
J         2
S         2
Q         2
Name: count, dtype: int64

In [7]:
classifier_df['annotation'].unique()

<StringArray>
['FP', 'R', 'A', 'j', 'N', 'V', 'Q', 'J', '/', 'L', 'F', 'a', 'E', 'f']
Length: 14, dtype: str

In [8]:

classifier_df_train = pd.read_pickle(
    CLASSIFIER_INPUT_PATH_TRAIN
)
classifier_df_val = pd.read_pickle(
    CLASSIFIER_INPUT_PATH_VAL
)
classifier_df_test = pd.read_pickle(
    CLASSIFIER_INPUT_PATH_TEST
)

# Keep the original variable available for cells that inspect one dataframe.
classifier_df = classifier_df_train

print("Train rows:", len(classifier_df_train))
print("Validation rows:", len(classifier_df_val))
print("Test rows:", len(classifier_df_test))

classifier_df_train


Train rows: 55686
Validation rows: 27291
Test rows: 28806


,record,reference_peak,alg2,alg3,alg4,alg5,annotation
0,232,31,-100,-100,-100,-100,FP
1,232,188,-100,-100,35,-100,FP
2,232,430,-100,-100,18,-100,FP
3,232,506,-13,-8,-100,-100,R
4,232,751,-12,-100,25,32,A
...,...,...,...,...,...,...,...
55681,114,648722,-12,-8,18,21,N
55682,114,649005,-11,-8,2,23,N
55683,114,649273,-11,-7,20,26,N
55684,114,649536,-11,-8,19,24,N


## Build or load classifier input

In [9]:
FULL_SEARCH_ROOT = REPO_ROOT / "Full_search_artifacts"

GRAPH_TRIALS = {}

trial_directories = sorted(
    (
        trial_dir
        for configuration_dir in FULL_SEARCH_ROOT.iterdir()
        if configuration_dir.is_dir()
        for trial_dir in configuration_dir.glob("trial_*")
        if trial_dir.is_dir()
    ),
    key=lambda path: (
        path.parent.name,
        int(path.name.removeprefix("trial_")),
    ),
)

for trial_directory in trial_directories:
    configuration_name = trial_directory.parent.name.lower()
    trial_number = int(trial_directory.name.removeprefix("trial_"))

    detector_name = (
        f"evolved_{configuration_name}_trial_{trial_number}"
    )

    GRAPH_TRIALS[detector_name] = trial_directory


print(f"Discovered {len(GRAPH_TRIALS)} evolved algorithms")

for detector_name, trial_directory in GRAPH_TRIALS.items():
    print(f"{detector_name}: {trial_directory}")

Discovered 36 evolved algorithms
evolved_early_na_trial_0: /Users/jeremiasz/code/ekg-evo/new/Full_search_artifacts/EARLY_NA/trial_000
evolved_early_na_trial_1: /Users/jeremiasz/code/ekg-evo/new/Full_search_artifacts/EARLY_NA/trial_001
evolved_early_na_trial_2: /Users/jeremiasz/code/ekg-evo/new/Full_search_artifacts/EARLY_NA/trial_002
evolved_early_na_trial_3: /Users/jeremiasz/code/ekg-evo/new/Full_search_artifacts/EARLY_NA/trial_003
evolved_early_na_trial_4: /Users/jeremiasz/code/ekg-evo/new/Full_search_artifacts/EARLY_NA/trial_004
evolved_early_na_trial_5: /Users/jeremiasz/code/ekg-evo/new/Full_search_artifacts/EARLY_NA/trial_005
evolved_early_others_trial_0: /Users/jeremiasz/code/ekg-evo/new/Full_search_artifacts/EARLY_OTHERS/trial_000
evolved_early_others_trial_1: /Users/jeremiasz/code/ekg-evo/new/Full_search_artifacts/EARLY_OTHERS/trial_001
evolved_early_others_trial_2: /Users/jeremiasz/code/ekg-evo/new/Full_search_artifacts/EARLY_OTHERS/trial_002
evolved_early_others_trial_3: /Use

In [10]:
existing_columns = set(classifier_df.columns)

graphs_already_present = [
    detector_name
    for detector_name in GRAPH_TRIALS
    if detector_name in existing_columns
]

graphs_to_add = [
    detector_name
    for detector_name in GRAPH_TRIALS
    if detector_name not in existing_columns
]

print("Already present evolved columns:")
for name in graphs_already_present:
    print(f"  {name}")

print("\nEvolved columns to add:")
for name in graphs_to_add:
    print(f"  {name}")

print(
    f"\nAlready present: {len(graphs_already_present)}, "
    f"to add: {len(graphs_to_add)}"
)

Already present evolved columns:

Evolved columns to add:
  evolved_early_na_trial_0
  evolved_early_na_trial_1
  evolved_early_na_trial_2
  evolved_early_na_trial_3
  evolved_early_na_trial_4
  evolved_early_na_trial_5
  evolved_early_others_trial_0
  evolved_early_others_trial_1
  evolved_early_others_trial_2
  evolved_early_others_trial_3
  evolved_early_others_trial_4
  evolved_early_others_trial_5
  evolved_early_v_trial_0
  evolved_early_v_trial_1
  evolved_early_v_trial_2
  evolved_early_v_trial_3
  evolved_early_v_trial_4
  evolved_early_v_trial_5
  evolved_late_na_trial_0
  evolved_late_na_trial_1
  evolved_late_na_trial_2
  evolved_late_na_trial_3
  evolved_late_na_trial_4
  evolved_late_na_trial_5
  evolved_late_others_trial_0
  evolved_late_others_trial_1
  evolved_late_others_trial_2
  evolved_late_others_trial_3
  evolved_late_others_trial_4
  evolved_late_others_trial_5
  evolved_late_v_trial_0
  evolved_late_v_trial_1
  evolved_late_v_trial_2
  evolved_late_v_trial_3
  

In [11]:

# Ordinary detector functions can also be added here.
NEW_DETECTORS = {
    # "alg6": algorithms.some_new_algorithm,
}

GRAPH_SETTINGS = {
    # Optional detector-specific settings:
    #
    # "evolved_early_na_trial_0": {
    #     "threshold_quantile": 0.98,
    #     "min_distance": 90,
    # },
}


split_configuration = {
    "train": (
        DATA_PATH_TRAIN,
        CLASSIFIER_INPUT_PATH_TRAIN,
    ),
    "val": (
        DATA_PATH_VAL,
        CLASSIFIER_INPUT_PATH_VAL,
    ),
    "test": (
        DATA_PATH_TEST,
        CLASSIFIER_INPUT_PATH_TEST,
    ),
}

source_dfs = {}
classifier_dfs = {}
missing_graph_names_by_split = {}

for split_name, (
    source_path,
    classifier_input_path,
) in split_configuration.items():
    source_dfs[split_name] = pd.read_pickle(source_path)
    classifier_dfs[split_name] = pd.read_pickle(
        classifier_input_path
    )

    missing_graph_names_by_split[split_name] = [
        detector_name
        for detector_name in GRAPH_TRIALS
        if detector_name not in classifier_dfs[split_name].columns
    ]

    existing_graph_names = [
        detector_name
        for detector_name in GRAPH_TRIALS
        if detector_name in classifier_dfs[split_name].columns
    ]

    print(f"\n{split_name.capitalize()} dataset")
    print(f"Discovered graphs: {len(GRAPH_TRIALS)}")
    print(f"Already present: {len(existing_graph_names)}")
    print(
        "Need to add: "
        f"{len(missing_graph_names_by_split[split_name])}"
    )


graph_names_to_load = sorted(
    {
        detector_name
        for names in missing_graph_names_by_split.values()
        for detector_name in names
    }
)

loaded_graphs = {}

for detector_name in graph_names_to_load:
    trial_directory = GRAPH_TRIALS[detector_name]
    _, graph = load_best_graph(trial_directory)
    loaded_graphs[detector_name] = graph


def add_detector_columns(
    split_name,
    source_df,
    classifier_df,
    classifier_input_path,
):
    """
    Add detector columns missing from one classifier dataframe.
    """
    detectors_to_add = {
        detector_name: detector
        for detector_name, detector in NEW_DETECTORS.items()
        if detector_name not in classifier_df.columns
    }

    detectors_to_add.update(
        {
            detector_name: loaded_graphs[detector_name]
            for detector_name in missing_graph_names_by_split[
                split_name
            ]
        }
    )

    if not detectors_to_add:
        print(
            f"\n{split_name.capitalize()}: "
            "no computation needed."
        )
        return classifier_df

    print(
        f"\n{split_name.capitalize()}: computing "
        f"{len(detectors_to_add)} new detector columns..."
    )

    # Rows must be generated using the same reference detector as before.
    update_detectors = {
        REFERENCE_DETECTOR: BASE_DETECTORS[
            REFERENCE_DETECTOR
        ],
        **detectors_to_add,
    }

    update_df = build_classifier_input(
        dataset=source_df,
        detectors=update_detectors,
        reference_detector=REFERENCE_DETECTOR,
        window=MATCH_WINDOW,
        graph_threshold_quantile=0.97,
        graph_min_distance=80,
        graph_settings=GRAPH_SETTINGS,
    )

    new_feature_names = list(detectors_to_add)
    key_columns = ["record", "reference_peak"]

    # Keep merge keys as integers in both dataframes.
    for column in key_columns:
        classifier_df[column] = pd.to_numeric(
            classifier_df[column],
            errors="raise",
        ).astype("int64")

        update_df[column] = pd.to_numeric(
            update_df[column],
            errors="raise",
        ).astype("int64")

    # Verify that the reference detector generated exactly the same rows.
    base_keys = pd.MultiIndex.from_frame(
        classifier_df[key_columns]
    )
    update_keys = pd.MultiIndex.from_frame(
        update_df[key_columns]
    )

    missing_keys = base_keys.difference(update_keys)
    extra_keys = update_keys.difference(base_keys)

    if len(missing_keys) > 0 or len(extra_keys) > 0:
        raise RuntimeError(
            "The reference-detector rows differ from the saved "
            f"{split_name} dataframe. "
            f"Missing rows: {len(missing_keys)}, "
            f"extra rows: {len(extra_keys)}. "
            "Rebuild the complete classifier dataframe instead."
        )

    columns_to_merge = [
        *key_columns,
        *new_feature_names,
        "annotation",
    ]

    update_subset = update_df[columns_to_merge].rename(
        columns={"annotation": "updated_annotation"}
    )

    classifier_df = classifier_df.merge(
        update_subset,
        on=key_columns,
        how="left",
        validate="one_to_one",
        sort=False,
    )

    # Verify that annotations have not changed.
    annotation_mismatch = (
        classifier_df["annotation"].astype(str)
        != classifier_df["updated_annotation"].astype(str)
    )

    if annotation_mismatch.any():
        raise RuntimeError(
            f"Annotation mismatch in "
            f"{annotation_mismatch.sum()} rows of the "
            f"{split_name} dataframe. "
            "Check MATCH_WINDOW and the source dataset."
        )

    if classifier_df[new_feature_names].isna().any().any():
        missing_counts = (
            classifier_df[new_feature_names]
            .isna()
            .sum()
        )
        missing_counts = missing_counts[
            missing_counts > 0
        ]

        raise RuntimeError(
            "Some new detector columns contain missing values:\n"
            f"{missing_counts}"
        )

    classifier_df = classifier_df.drop(
        columns="updated_annotation"
    )

    # Preserve integer detector outputs.
    classifier_df[new_feature_names] = (
        classifier_df[new_feature_names]
        .astype(int)
    )

    classifier_df.to_pickle(classifier_input_path)

    print(f"Saved to: {classifier_input_path}")

    return classifier_df


for split_name, (
    _,
    classifier_input_path,
) in split_configuration.items():
    classifier_dfs[split_name] = add_detector_columns(
        split_name=split_name,
        source_df=source_dfs[split_name],
        classifier_df=classifier_dfs[split_name],
        classifier_input_path=classifier_input_path,
    )


classifier_df_train = classifier_dfs["train"]
classifier_df_val = classifier_dfs["val"]
classifier_df_test = classifier_dfs["test"]

# Keep the original variable available for cells that inspect one dataframe.
classifier_df = classifier_df_train

display(classifier_df_train.head())



Train dataset
Discovered graphs: 36
Already present: 0
Need to add: 36

Val dataset
Discovered graphs: 36
Already present: 0
Need to add: 36

Test dataset
Discovered graphs: 36
Already present: 0
Need to add: 36

Train: computing 36 new detector columns...
Saved to: /Users/jeremiasz/code/ekg-evo/new/classifier/classifier_input_train.pickle

Val: computing 36 new detector columns...
Saved to: /Users/jeremiasz/code/ekg-evo/new/classifier/classifier_input_val.pickle

Test: computing 36 new detector columns...
Saved to: /Users/jeremiasz/code/ekg-evo/new/classifier/classifier_input_test.pickle


,record,reference_peak,alg2,alg3,alg4,alg5,annotation,evolved_early_na_trial_0,evolved_early_na_trial_1,evolved_early_na_trial_2,...,evolved_late_others_trial_2,evolved_late_others_trial_3,evolved_late_others_trial_4,evolved_late_others_trial_5,evolved_late_v_trial_0,evolved_late_v_trial_1,evolved_late_v_trial_2,evolved_late_v_trial_3,evolved_late_v_trial_4,evolved_late_v_trial_5
0,232,31,-100,-100,-100,-100,FP,-100,-100,-100,...,-100,-100,-100,-100,-100,-100,-100,-100,-100,-100
1,232,188,-100,-100,35,-100,FP,-100,-100,-100,...,-100,-100,-100,-100,-100,-100,-100,-100,-100,-100
2,232,430,-100,-100,18,-100,FP,-100,-100,-100,...,-100,-100,43,-100,-100,-100,-100,-100,-100,-100
3,232,506,-13,-8,-100,-100,R,-19,-16,-17,...,-21,-25,-33,-24,-18,-19,-18,-19,-19,-22
4,232,751,-12,-100,25,32,A,-18,-15,-17,...,-26,-25,-32,-24,-16,-19,-18,-19,-18,-22


In [16]:

classifier_df_train = pd.read_pickle(
    CLASSIFIER_INPUT_PATH_TRAIN
)
classifier_df_val = pd.read_pickle(
    CLASSIFIER_INPUT_PATH_VAL
)
classifier_df_test = pd.read_pickle(
    CLASSIFIER_INPUT_PATH_TEST
)

SELECTED_FEATURES = [
    # Existing algorithms
    "alg2",
    "alg3",
    "alg4",
    "alg5",

    # All automatically discovered evolved graphs
    *GRAPH_TRIALS.keys(),
]

# SELECTED_FEATURES = ORDERED_FEATURE_LIST[:20]
# SELECTED_FEATURES = [
#     "evolved_early_na_trial_3",
#     "alg3",
#     "evolved_early_v_trial_0",
#     "evolved_early_v_trial_3",
#     "alg5",
#     "evolved_early_v_trial_2",
#     "evolved_early_others_trial_1",
# ]

if REFERENCE_DETECTOR in SELECTED_FEATURES:
    raise ValueError(
        "The reference detector cannot be used as a feature because "
        "its detections define the dataframe rows"
    )

for split_name, split_df in {
    "train": classifier_df_train,
    "val": classifier_df_val,
    "test": classifier_df_test,
}.items():
    missing_features = [
        feature
        for feature in SELECTED_FEATURES
        if feature not in split_df.columns
    ]

    if missing_features:
        raise KeyError(
            f"These features are not present in the "
            f"{split_name} classifier input: "
            f"{missing_features}"
        )


# LABEL_GROUPING = "current"
# LABEL_GROUPING = "aami"


def prepare_classifier_split(
    classifier_df,
    split_name,
):
    X = (
        classifier_df[SELECTED_FEATURES]
        .astype(int)
        .copy()
    )

    y = group_annotations(
        classifier_df["annotation"],
        scheme=LABEL_GROUPING,
    )

    # Ambiguous matches are not reliable targets.
    ambiguous_mask = y == "AMB"

    if ambiguous_mask.any():
        print(
            f"Removing ambiguous detections from "
            f"{split_name}: {ambiguous_mask.sum()}"
        )
        X = X.loc[~ambiguous_mask].copy()
        y = y.loc[~ambiguous_mask].copy()

    return X, y


X_train, y_train = prepare_classifier_split(
    classifier_df_train,
    "train",
)

X_val, y_val = prepare_classifier_split(
    classifier_df_val,
    "validation",
)

X_test, y_test = prepare_classifier_split(
    classifier_df_test,
    "test",
)

print("Label grouping:", LABEL_GROUPING)
print("Features:", SELECTED_FEATURES)
print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Test rows:", len(X_test))

print("\nTraining labels:")
display(y_train.value_counts())

print("\nValidation labels:")
display(y_val.value_counts())

print("\nTest labels:")
display(y_test.value_counts())


Label grouping: current
Features: ['alg2', 'alg3', 'alg4', 'alg5', 'evolved_early_na_trial_0', 'evolved_early_na_trial_1', 'evolved_early_na_trial_2', 'evolved_early_na_trial_3', 'evolved_early_na_trial_4', 'evolved_early_na_trial_5', 'evolved_early_others_trial_0', 'evolved_early_others_trial_1', 'evolved_early_others_trial_2', 'evolved_early_others_trial_3', 'evolved_early_others_trial_4', 'evolved_early_others_trial_5', 'evolved_early_v_trial_0', 'evolved_early_v_trial_1', 'evolved_early_v_trial_2', 'evolved_early_v_trial_3', 'evolved_early_v_trial_4', 'evolved_early_v_trial_5', 'evolved_late_na_trial_0', 'evolved_late_na_trial_1', 'evolved_late_na_trial_2', 'evolved_late_na_trial_3', 'evolved_late_na_trial_4', 'evolved_late_na_trial_5', 'evolved_late_others_trial_0', 'evolved_late_others_trial_1', 'evolved_late_others_trial_2', 'evolved_late_others_trial_3', 'evolved_late_others_trial_4', 'evolved_late_others_trial_5', 'evolved_late_v_trial_0', 'evolved_late_v_trial_1', 'evolved_la

annotation
N        35809
L         4607
FP        4254
R         4068
V         2822
/         2394
A         1511
Other      221
Name: count, dtype: int64


Validation labels:


annotation
N        18513
L         2003
V         1937
/         1283
R         1249
FP        1113
Other     1065
A          128
Name: count, dtype: int64


Test labels:


annotation
N        18955
V         2021
R         1910
/         1542
L         1456
Other     1068
FP         974
A          880
Name: count, dtype: int64

## Prepare labels and split the data

In [17]:

X_train, y_train = prepare_classifier_split(
    classifier_df_train,
    "train",
)

X_val, y_val = prepare_classifier_split(
    classifier_df_val,
    "validation",
)

X_test, y_test = prepare_classifier_split(
    classifier_df_test,
    "test",
)

print("Features:", SELECTED_FEATURES)
print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Test rows:", len(X_test))

print("\nTraining labels:")
display(y_train.value_counts())

print("\nValidation labels:")
display(y_val.value_counts())

print("\nTest labels:")
display(y_test.value_counts())


Features: ['alg2', 'alg3', 'alg4', 'alg5', 'evolved_early_na_trial_0', 'evolved_early_na_trial_1', 'evolved_early_na_trial_2', 'evolved_early_na_trial_3', 'evolved_early_na_trial_4', 'evolved_early_na_trial_5', 'evolved_early_others_trial_0', 'evolved_early_others_trial_1', 'evolved_early_others_trial_2', 'evolved_early_others_trial_3', 'evolved_early_others_trial_4', 'evolved_early_others_trial_5', 'evolved_early_v_trial_0', 'evolved_early_v_trial_1', 'evolved_early_v_trial_2', 'evolved_early_v_trial_3', 'evolved_early_v_trial_4', 'evolved_early_v_trial_5', 'evolved_late_na_trial_0', 'evolved_late_na_trial_1', 'evolved_late_na_trial_2', 'evolved_late_na_trial_3', 'evolved_late_na_trial_4', 'evolved_late_na_trial_5', 'evolved_late_others_trial_0', 'evolved_late_others_trial_1', 'evolved_late_others_trial_2', 'evolved_late_others_trial_3', 'evolved_late_others_trial_4', 'evolved_late_others_trial_5', 'evolved_late_v_trial_0', 'evolved_late_v_trial_1', 'evolved_late_v_trial_2', 'evolved_

annotation
N        35809
L         4607
FP        4254
R         4068
V         2822
/         2394
A         1511
Other      221
Name: count, dtype: int64


Validation labels:


annotation
N        18513
L         2003
V         1937
/         1283
R         1249
FP        1113
Other     1065
A          128
Name: count, dtype: int64


Test labels:


annotation
N        18955
V         2021
R         1910
/         1542
L         1456
Other     1068
FP         974
A          880
Name: count, dtype: int64

## Train and tune the decision tree

In [18]:
parameter_grid = {
    "criterion": ["entropy"],
    "max_depth": [15],
    # "min_samples_split": [2, 5, 10],
    # "min_samples_leaf": [1, 2, 5],
    "class_weight": ["balanced"],
    # "ccp_alpha": [0.0, 0.001, 0.01],
}

cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=0,
)

search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=0),
    param_grid=parameter_grid,
    scoring="accuracy",
    cv=cross_validation,
    n_jobs=-1,
)

search.fit(X_train, y_train)
classifier = search.best_estimator_

train_predictions = classifier.predict(X_train)
test_predictions = classifier.predict(X_test)

print("Best parameters:", search.best_params_)
print(f"Best CV macro F1: {search.best_score_:.4f}")
print(f"Training accuracy: {accuracy_score(y_train, train_predictions):.4f}")
print(f"Test accuracy: {accuracy_score(y_test, test_predictions):.4f}")
print()
print(classification_report(y_test, test_predictions, digits=4, zero_division=0))

Best parameters: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 15}
Best CV macro F1: 0.9317
Training accuracy: 0.9758
Test accuracy: 0.5004

              precision    recall  f1-score   support

           /     0.0026    0.0006    0.0010      1542
           A     0.1852    0.0909    0.1220       880
          FP     0.1812    0.6222    0.2806       974
           L     0.0000    0.0000    0.0000      1456
           N     0.7811    0.6710    0.7219     18955
       Other     0.0524    0.0094    0.0159      1068
           R     0.0423    0.0152    0.0223      1910
           V     0.1622    0.4800    0.2425      2021

    accuracy                         0.5004     28806
   macro avg     0.1759    0.2362    0.1758     28806
weighted avg     0.5420    0.5004    0.5074     28806



In [19]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_score


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

selected_features = []
remaining_features = SELECTED_FEATURES.copy()
selection_history = []

for step in range(7):
    step_results = []

    for candidate in remaining_features:
        candidate_features = selected_features + [candidate]

        model = clone(classifier)

        scores = cross_val_score(
            model,
            X_train[candidate_features],
            y_train,
            scoring="balanced_accuracy",
            cv=cv,
            n_jobs=-1,
        )

        step_results.append(
            {
                "candidate": candidate,
                "features": candidate_features,
                "mean_balanced_accuracy": scores.mean(),
                "std_balanced_accuracy": scores.std(),
            }
        )

    step_results_df = (
        pd.DataFrame(step_results)
        .sort_values(
            "mean_balanced_accuracy",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    best_candidate = step_results_df.loc[0, "candidate"]

    selected_features.append(best_candidate)
    remaining_features.remove(best_candidate)

    selection_history.append(step_results_df)

    print(
        f"Step {step + 1}: added {best_candidate} | "
        f"balanced accuracy = "
        f"{step_results_df.loc[0, 'mean_balanced_accuracy']:.4f}"
    )

BEST_4_FEATURES = selected_features

print("\nSelected features:")
print(BEST_4_FEATURES)

Step 1: added evolved_early_na_trial_3 | balanced accuracy = 0.6129
Step 2: added alg5 | balanced accuracy = 0.7654
Step 3: added alg3 | balanced accuracy = 0.8089
Step 4: added evolved_early_v_trial_3 | balanced accuracy = 0.8334
Step 5: added evolved_early_na_trial_4 | balanced accuracy = 0.8419
Step 6: added evolved_early_v_trial_5 | balanced accuracy = 0.8477
Step 7: added alg2 | balanced accuracy = 0.8487

Selected features:
['evolved_early_na_trial_3', 'alg5', 'alg3', 'evolved_early_v_trial_3', 'evolved_early_na_trial_4', 'evolved_early_v_trial_5', 'alg2']


## Confusion matrix

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    recall_score,
    multilabel_confusion_matrix,
)

classes = classifier.classes_

# Overall metrics
accuracy = accuracy_score(
    y_test,
    test_predictions,
)

balanced_accuracy = balanced_accuracy_score(
    y_test,
    test_predictions,
)

# Sensitivity = recall for each class
sensitivities = recall_score(
    y_test,
    test_predictions,
    labels=classes,
    average=None,
    zero_division=0,
)

# One-vs-rest confusion matrix for each class
class_confusion_matrices = multilabel_confusion_matrix(
    y_test,
    test_predictions,
    labels=classes,
)

specificities = []

for class_matrix in class_confusion_matrices:
    tn, fp, fn, tp = class_matrix.ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    specificities.append(specificity)
# Plot
fig, ax = plt.subplots(figsize=(11, 9))

display_object = ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    labels=classes,
    cmap="Blues",
    values_format="d",
    ax=ax,
    colorbar=True,
)

# Numbers inside confusion-matrix cells
for text in display_object.text_.ravel():
    text.set_fontsize(15)
    text.set_fontweight("bold")

# Title
ax.set_title(
    "Best combination of 7 detectors",
    # "5 detectors from literature",
    # "All 41 detectors",
    fontsize=22,
    fontweight="bold",
    pad=20,
)

# Axis descriptions
ax.set_xlabel(
    "Predicted label",
    fontsize=18,
    labelpad=12,
)

ax.set_ylabel(
    "True label",
    fontsize=18,
    labelpad=12,
)

# Class names on both axes
ax.tick_params(
    axis="both",
    labelsize=16,
)

# Colorbar numbers
if display_object.im_.colorbar is not None:
    display_object.im_.colorbar.ax.tick_params(
        labelsize=14,
    )

plt.tight_layout()
plt.show()

# Metrics table
metrics_df = pd.DataFrame(
    {
        "class": classes,
        "sensitivity": sensitivities,
        "specificity": specificities,
    }
)

overall_metrics_df = pd.DataFrame(
    {
        "metric": [
            "accuracy",
            "balanced_accuracy",
        ],
        "value": [
            accuracy,
            balanced_accuracy,
        ],
    }
)

display(overall_metrics_df)
display(metrics_df)

## Tree and feature importance

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))

plot_tree(
    classifier,
    feature_names=SELECTED_FEATURES,
    class_names=[str(label) for label in classifier.classes_],
    filled=True,
    rounded=True,
    max_depth=4,
    fontsize=8,
    ax=ax,
)

ax.set_title("Decision tree — first four levels")
plt.tight_layout()
plt.show()

feature_importance = (
    pd.Series(
        classifier.feature_importances_,
        index=SELECTED_FEATURES,
        name="importance",
    )
    .sort_values(ascending=False)
    .to_frame()
)

display(feature_importance)